## Functionalility Tests for JSON Handling

In [ ]:
import sys
from pathlib import Path

# Get the notebook's directory and navigate up to the project root
notebook_dir = Path().resolve()
project_root = notebook_dir.parent.parent  # Goes up two levels from notebooks/functionality_tests to grafa

# Add project root to Python path if not already there
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [ ]:
from grafa.document.load.input_format import process_json

In [ ]:
result = await process_json("/Users/Pablo.Vargas2/Documents/grafa/notebooks/testing_data/test.json")

In [ ]:
result

### Now setup the whole ingestion pipeline for testing

In [ ]:
import os
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from grafa.client import GrafaClient, GrafaConfig
from dotenv import load_dotenv
load_dotenv()

In [ ]:
import logging
import warnings

for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

#Basic logging setup for Jupyter notebooks
logging.basicConfig(
    level=logging.INFO,
    format='%(levelname)s - %(name)s - %(message)s',
    force=True  
)

# Set Grafa client logger to show detailed info
grafa_logger = logging.getLogger('grafa.client')
#grafa_logger.setLevel(logging.INFO)
grafa_logger.setLevel(logging.DEBUG)
# Prevent propagation to avoid duplicates
grafa_logger.propagate = True

# Suppress Neo4j notifications (they're just performance info, not errors)
neo4j_logger = logging.getLogger('neo4j.notifications')
neo4j_logger.setLevel(logging.WARNING)  # Only show warnings and errors, not info


print("✓ Logging configured - duplicates prevented, Neo4j notifications suppressed")

In [ ]:
# Create embedding and LLM objects
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")
embedding_dimension = 1536  # or 1024 for some models
llm = ChatOpenAI(model="gpt-5-chat-latest", temperature=0, max_tokens=4096)

grafa_config = await GrafaConfig.create(
        embedding_model=embedding_model,
        embedding_dimension=embedding_dimension,
        semantic_similarity_function="cosine",
        llm=llm,
        neo4j_driver=None,  # will be set automatically
        local_storage_path="/tmp/data",     # will be set automatically
    )

In [ ]:
client = await GrafaClient.from_yaml(
        yaml_path="/Users/Pablo.Vargas2/Documents/grafa/testing.yaml",
        db_name="my_db",  # This will create the database in Neo4j
        grafa_config=grafa_config,
    )

In [ ]:
document, chunks, entities, relationships = await client.ingest_file(
    document_name="testing_json",
    document_path="/Users/Pablo.Vargas2/Documents/grafa/notebooks/testing_data/test.json",
    context="Json document",
    author="Pablo"
)